In [12]:
import xarray as xr
import numpy as np
import cmcrameri.cm as cmc
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data=xr.open_mfdataset("/wolke_scratch/dnikolo/CLAAS_Data/Resampled_Data/np/2010/01/01/Agg_03*")


In [3]:
ctx_data=xr.open_mfdataset("/wolke_scratch/dnikolo/CLAAS_Data/np/2010/01/01/CTX*")

In [28]:
cpp_data=xr.open_mfdataset("/wolke_scratch/dnikolo/CLAAS_Data/np/2010/01/01/CPP*")

In [30]:
print(ctx_data["ctp"].min().values,ctx_data["ctp"].max().values)
print(ctx_data["cth"].min().values,ctx_data["cth"].max().values)
print(cpp_data["cot"].min().values,cpp_data["cot"].max().values)
print(cpp_data["cwp"].min().values,cpp_data["cwp"].max().values)

70.0 1143.9
0.0 18358.0
0.0 150.0
0.0 3.2766


In [38]:
minv, maxv = 235.15, 273.15
minv_ctt,maxv_ctt=235.15,273.15
# minv_ctp,maxv_ctp=0,11000
# minv_cth,maxv_cth=0, 30000
minv_ctp,maxv_ctp= 70.0, 1143.9
minv_cth,maxv_cth= 0.0,18358.0
minv_cot, maxv_cot = 0.0, 150.0
minv_cwp, maxv_cwp = 0.0, 3.2766
brightness_coverage=0.7
idx_to_generate = [i for i in range(0,96) if ((i<10) ^ (i%10==0))]
idx_to_generate.append(48)
print(idx_to_generate)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30, 40, 50, 60, 70, 80, 90, 48]


In [44]:

for frame_idx in idx_to_generate:
    arr = data['ctt_resampled'].isel(time=frame_idx).values
    # now imsave will map vmin→colormap bottom, vmax→colormap top
    mask = (arr<maxv_ctt) & (arr>minv_ctt)
#   mask =np.repeat(mask[ np.newaxis, :, :], 3, axis=0)
    arr[mask] = np.nan
    plt.imsave(f'/wolke_scratch/dnikolo/Glaciation_time_estimator/Prototyping_files/Tracking/Frame_vis/ctt_agg_fr{frame_idx}_flare.jpg',
            arr,
            cmap=sns.color_palette("flare", as_cmap=True),
            vmin=235.25,
            vmax=273.15)
    plt.imsave(f'/wolke_scratch/dnikolo/Glaciation_time_estimator/Prototyping_files/Tracking/Frame_vis/ctt_agg_fr{frame_idx}_viridis.jpg',
            arr,
            cmap="viridis",
            vmin=235.25,
            vmax=273.15)
    plt.imsave(f'/wolke_scratch/dnikolo/Glaciation_time_estimator/Prototyping_files/Tracking/Frame_vis/ctt_agg_fr{frame_idx}_cubehelix.jpg',
            arr,
            cmap=sns.cubehelix_palette(as_cmap=True),
            vmin=235.25,
            vmax=273.15)
    plt.imsave(f'/wolke_scratch/dnikolo/Glaciation_time_estimator/Prototyping_files/Tracking/Frame_vis/ctt_agg_fr{frame_idx}_rainbow.jpg',
            arr,
            cmap="rainbow",
            vmin=235.25,
            vmax=273.15)
    arr_norm = (arr - minv_ctt) / (maxv_ctt - minv_ctt)
    arr_uint8 = (arr_norm * 255*(brightness_coverage) + 255*(1-brightness_coverage)).astype(np.uint8)
    # 3. build a grayscale image (or convert to RGB if you like)
    img = Image.fromarray(arr_uint8, mode='L')
    img.save(f'/wolke_scratch/dnikolo/Glaciation_time_estimator/Prototyping_files/Tracking/Frame_vis/ctt_agg_fr{frame_idx}_gray.jpg', quality=100)

/tmp/ipykernel_42925/2434890678.py:28: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = (arr_norm * 255*(brightness_coverage) + 255*(1-brightness_coverage)).astype(np.uint8)


In [47]:
from PIL import Image
import numpy as np
# concat the three DataArrays along a new dim "band"
da = xr.concat(
    [ctx_data["ctt"], ctx_data["ctp"], ctx_data["cth"]],
    dim="band"
)

for frame_idx in idx_to_generate: 
    # 1. grab your array
    
    arr = da.isel(time=frame_idx).values
    
    mask = (arr[0]<maxv_ctt) & (arr[0]>minv_ctt)
    mask =np.repeat(mask[ np.newaxis, :, :], 3, axis=0)
    arr[mask] = np.nan
    # arr=arr[mask]
    # print(arr.shape)
    # 2. normalize into [0–255] uint8
    arr_norm = np.empty_like(arr)
    
    arr_norm[0] = (arr[0] - minv_ctt) / (maxv_ctt - minv_ctt)
    arr_norm[1] = (arr[1] - minv_ctp) / (maxv_ctp - minv_ctp)
    arr_norm[2] = (arr[2] - minv_cth) / (maxv_cth - minv_cth)
    arr_uint8 = (arr_norm * 255*(brightness_coverage) + 255*(1-brightness_coverage)).astype(np.uint8)
    
    # # 3. build a grayscale image (or convert to RGB if you like)
    img = Image.fromarray(np.transpose(arr_uint8,(1,2,0)), mode='RGB')
    img.save(f'/wolke_scratch/dnikolo/Glaciation_time_estimator/Prototyping_files/Tracking/Frame_vis/comb_tph_full_fr{frame_idx}.jpg', quality=100)
arr_uint8.shape

/tmp/ipykernel_42925/3047806209.py:25: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = (arr_norm * 255*(brightness_coverage) + 255*(1-brightness_coverage)).astype(np.uint8)


(3, 896, 3133)

In [46]:
from PIL import Image
import numpy as np
# concat the three DataArrays along a new dim "band"
da_tow = xr.concat(
    [ctx_data["ctt"], cpp_data["cot"], cpp_data["cwp"]],
    dim="band"
)

for frame_idx in [48]: 
    # 1. grab your array
    
    arr = da.isel(time=frame_idx).values
    
    mask = (arr[0]<maxv_ctt) & (arr[0]>minv_ctt)
    mask =np.repeat(mask[ np.newaxis, :, :], 3, axis=0)
    arr[mask] = np.nan
    # arr=arr[mask]
    # print(arr.shape)
    # 2. normalize into [0–255] uint8
    arr_norm = np.empty_like(arr)
    
    arr_norm[0] = (arr[0] - minv_ctt) / (maxv_ctt - minv_ctt)
    arr_norm[1] = (arr[1] - minv_cot) / (maxv_ctp - minv_cot)
    arr_norm[2] = (arr[2] - minv_cwp) / (maxv_cth - minv_cwp)
    arr_uint8 = (arr_norm * 255*(brightness_coverage) + 255*(1-brightness_coverage)).astype(np.uint8)
    
    # # 3. build a grayscale image (or convert to RGB if you like)
    img = Image.fromarray(np.transpose(arr_uint8,(1,2,0)), mode='RGB')
    img.save(f'/wolke_scratch/dnikolo/Glaciation_time_estimator/Prototyping_files/Tracking/Frame_vis/comb_tow_full_fr{frame_idx}.jpg', quality=100)
arr_uint8.shape

/tmp/ipykernel_42925/3534712943.py:25: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = (arr_norm * 255*(brightness_coverage) + 255*(1-brightness_coverage)).astype(np.uint8)


(3, 896, 3133)

In [87]:
print(np.median(arr_uint8[0][arr_uint8[0]!=0]),np.median(arr_uint8[1][arr_uint8[1]!=0]),np.median(arr_uint8[2][arr_uint8[2]!=0]))

135.0 107.0 68.0


In [18]:
plt.close()# .values.tofile("ctt_binary_agg.npy")